In [60]:
import torch
import torch.nn as nn 

from torch.utils.data import TensorDataset, DataLoader


### Generated data

In [62]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()
X = data.data
y = data.target

df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y
print(df.head())


   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0             

Split into train and val

In [65]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("train X_train.shape:", X_train.shape)
print("train y_train.shape:", y_train.shape)

print("val X_val.shape:", X_val.shape)
print("val y_val.shape:", y_val.shape)

train X_train.shape: (455, 30)
train y_train.shape: (455,)
val X_val.shape: (114, 30)
val y_val.shape: (114,)


Scale and convert to tensors

In [66]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

#convert numpy to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train) #wrap the tensors into a dataset
valid_dataset = TensorDataset(X_val, y_val) #wrap the tensors into a dataset

In [67]:

#create a data loader for batching and shuffling
train_loader = DataLoader(
    train_dataset, 
    batch_size=16, 
    shuffle=True)  #shuffle means shuffling order each epoch

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16, 
    shuffle=False
)


### Train and eval

In [68]:
model = nn.Linear(30,2) #30 input features, 2 possible output classes

criterion = nn.CrossEntropyLoss() #loss function for multi-class classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) #optimizer for updating model parameters



In [69]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
# we want all computations to be done on the same device (CPU, GPU, or MPS)
model = model.to(device)

model.train() #activates traiing mode (enable dropout, batchnorm)

Linear(in_features=30, out_features=2, bias=True)

In [ ]:
for epoch in range(10):

    # =====================
    # TRAINING
    # =====================
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    avg_train_loss = train_loss / len(train_loader)


    # =====================
    # VALIDATION
    # =====================
    model.eval()

    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in valid_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

            predicted = outputs.argmax(dim=1)

            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()


    avg_val_loss = val_loss / len(valid_loader)
    accuracy = correct / total

    print(
        f"Epoch {epoch+1}: "
        f"Train Loss: {avg_train_loss:.4f}, "
        f"Val Loss = {avg_val_loss:.4f}, "
        f"Val Accuracy = {accuracy:.4f}"
    )

Epoch 1: Val Loss = 0.1243, Val Accuracy = 0.9561
Epoch 2: Val Loss = 0.1043, Val Accuracy = 0.9561
Epoch 3: Val Loss = 0.0979, Val Accuracy = 0.9737
Epoch 4: Val Loss = 0.0904, Val Accuracy = 0.9737
Epoch 5: Val Loss = 0.0866, Val Accuracy = 0.9737
Epoch 6: Val Loss = 0.0858, Val Accuracy = 0.9649
Epoch 7: Val Loss = 0.0837, Val Accuracy = 0.9737
Epoch 8: Val Loss = 0.0801, Val Accuracy = 0.9649
Epoch 9: Val Loss = 0.0780, Val Accuracy = 0.9649
Epoch 10: Val Loss = 0.0760, Val Accuracy = 0.9737


In [ ]:
# for epoch in range(10):
#     for X_batch, y_batch in train_loader:



#         X_batch = X_batch.to(device)
#         y_batch = y_batch.to(device)

#         optimizer.zero_grad()

#         outputs = model(X_batch) #forward pass
#         loss = criterion(outputs, y_batch)

#         loss.backward() #backprop: compute gradients
#         optimizer.step() #update model param

### Eval mode

In [ ]:

# model.eval()

# val_loss = 0.0
# correct = 0
# total = 0

# with torch.no_grad(): #disable gradient computation for validation
#     for X_batch, y_batch in valid_loader: 
        
#         X_batch = X_batch.to(device)
#         y_batch = y_batch.to(device)

#         outputs = model(X_batch) #forward pass 
#         loss = criterion(outputs, y_batch) 

#         # EVALUATION METRICS
#         #store loss for this batch
#         val_loss += loss.item() # .item() extracts the number from the loss tensor
        
                                
#         #class predictions: gives idx of the class w highest logit
#         predicted = outputs.argmax(dim=1) 

#         #for accuracy calculation
#         total += y_batch.size(0)
#         correct += (predicted == y_batch).sum().item()

# accuracy = correct / total
# avg_val_loss = val_loss / len(valid_loader)

# print("validation loss:", avg_val_loss)
# print("validation accuracy:", accuracy)


validation loss: 0.0779002197086811
validation accuracy: 0.9736842105263158
